In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Analyzing Month-by-Month POI Openings & Closings with Places Insights Historical Data

<table align="left">
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Fplaces_insights%2Fnotebooks%2Fh3_historical_data_demo%2Fplaces_insights_historical_H3_demo.ipynb?utm_source=places_insights_notebooks">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/bigquery/import?url=https%3A%2F%2Fgithub.com%2Fgooglemaps-samples%2Finsights-samples%2Fblob%2Fmain%2Fplaces_insights%2Fnotebooks%2Fh3_historical_data_demo%2Fplaces_insights_historical_H3_demo.ipynb&utm_source=places_insights_notebooks">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/googlemaps-samples/insights-samples/blob/main/places_insights/notebooks/h3_historical_data_demo/places_insights_historical_H3_demo.ipynb">
      <img width="32px" src="https://upload.wikimedia.org/wikipedia/commons/9/91/Octicons-mark-github.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<br><br><br>

> **⚠️ Important Requirement:** Before executing this notebook, ensure your Google Cloud project is subscribed to the relevant **Places Insights** country or sample dataset via Analytics Hub (defaulted to Great Britain: `places_insights___gb`) and that the **Map Tiles API** and **Places API (New)** are enabled on your Google Maps Platform API key. See [Set up Places Insights](https://developers.google.com/maps/documentation/placesinsights/cloud-setup) for onboarding instructions.

### Overall Goal

Static point-in-time POI counts often mask underlying commercial turnover: if a neighborhood has 25 operational establishments in January and 25 in August, a net count delta of `0` hides whether the market is stagnant or experiencing rapid retail churn (for example, 4 existing businesses closing while 4 new competitors open).

In this notebook, we use **Places Insights Historical Data** and the **`PLACES_COUNT_PER_H3`** function with the **`snapshot_date`** parameter to track **month-by-month POI openings, closings, and net growth across a target city at H3 spatial resolution**. While the notebook defaults to analyzing **`coffee_shop` locations across London, UK at H3 Resolution 9**, you can customize the **target country dataset, city coordinates, place type, H3 resolution, and historical date range** directly in **Step 2**.

### Key Technologies Used

*   **[Places Insights (`PLACES_COUNT_PER_H3`)](https://developers.google.com/maps/documentation/placesinsights/place-count-functions/places-count-per-h3):** Aggregates historical POI counts and up to 250 `sample_place_ids` per H3 hexagonal cell for any monthly `snapshot_date` back to January 2024.
*   **[Google Maps Map Tiles API (2D Tiles)](https://developers.google.com/maps/documentation/tile/2d-tiles-overview):** Provides official Google Maps road basemap tiles and dynamic viewport copyright attribution for Folium visualizations.
*   **[Google Maps Places API (New)](https://developers.google.com/maps/documentation/places/web-service/place-details):** Resolves `opened_place_ids` and `closed_place_ids` into ground-truth storefront names, addresses, and ratings.
*   **[BigQuery](https://cloud.google.com/bigquery):** Executes spatial H3 indexing and array set-difference window functions (`LAG()` + `UNNEST()`).
*   **Python Libraries:** **GeoPandas** (spatial geometry handling), **Folium** (interactive H3 choropleth mapping), **Plotly** (month-by-month diverging bar & trend charts), and **Requests** (REST API handshakes).

*Note: This notebook executes queries and API calls that incur costs. See [Google Maps Platform Pricing](https://mapsplatform.google.com/pricing/) and [BigQuery Pricing](https://cloud.google.com/bigquery/pricing) for details.*

### The Step-by-Step Workflow

1.  **Setup & Dual-Environment Authentication:** Authenticate to Google Cloud and load `GCP_PROJECT_ID` and `GMP_API_KEY` across either Consumer Colab or Enterprise/Local Jupyter.
2.  **Pipeline Configuration (Customizable Location & Place Type):** Configure the target dataset, location center/radius, place type (defaulted to London & `coffee_shop`), H3 resolution (`9`), and historical month window.
3.  **Google Maps 2D Tiles Session Handshake:** Initialize an authenticated 2D Map Tiles session, fetch live viewport attribution, and prepare official Google logo assets.
4.  **Historical H3 Snapshot & Churn Query (`PLACES_COUNT_PER_H3`):** Query monthly snapshots via `PLACES_COUNT_PER_H3` and compute month-over-month set differences on `sample_place_ids` per `h3_cell_index`.
5.  **Month-by-Month Trend Chart:** Plot a diverging bar and trend line chart contrasting monthly POI openings (`+`), closings (`-`), and net growth.
6.  **Interactive H3 Choropleth Map with Map Key:** Render an interactive Folium H3 map with an embedded visual Map Key over Google Maps 2D Tiles to inspect neighborhood-level openings and closings.
7.  **Ground-Truth Place Details Lookup:** Pass `opened_place_ids` and `closed_place_ids` from high-turnover H3 cells to the Places API (New) to inspect real storefront names and addresses.

### How to Use This Notebook

1.  **Prerequisites & Secrets:** Configure the following keys in the Colab "Secrets" tab (the **key icon** on the left menu):
    *   `GCP_PROJECT_ID`: Your Google Cloud Project ID authorized for the target APIs/datasets.
    *   `GMP_API_KEY`: A Google Maps Platform API key with **Map Tiles API** and **Places API (New)** enabled.
2.  **Customize Your Target Location & Place Type (Optional):** In **Step 2 (`Pipeline Configuration`)**, keep the default settings (**London, UK** and **`coffee_shop`**) or update `TARGET_LOCATION_NAME`, `LINKED_DATASET`, `CENTER_LAT`, `CENTER_LNG`, `SEARCH_RADIUS_METERS`, and `PLACE_TYPE` to analyze any other city or category.
3.  **Run the Cells:** Execute the cells in sequence from top to bottom.

In [ ]:
# @title 1. Setup & Authentication
# @markdown Authenticate to Google Cloud, retrieve secrets, and initialize the BigQuery client.
import sys
import getpass
import requests
import pandas as pd
import geopandas as gpd
from shapely import wkt
from google.cloud import bigquery
import google.auth

try:
    # --- ATTEMPT 1: Standard Consumer Colab ---
    from google.colab import userdata, auth

    # Attempt to retrieve secrets from the left-hand sidebar
    GCP_PROJECT_ID = userdata.get('GCP_PROJECT_ID')
    GMP_API_KEY = userdata.get('GMP_API_KEY')
    print("✅ Standard Colab Secrets found.")

    # Authenticate via user pop-up
    print("🔄 Authenticating user...")
    auth.authenticate_user(project_id=GCP_PROJECT_ID)

    # Initialize BigQuery client
    client = bigquery.Client(project=GCP_PROJECT_ID)
    print(f"✅ Authenticated to Google Cloud Project: {GCP_PROJECT_ID}")

except (ImportError, Exception) as e:
    # --- ATTEMPT 2: Colab Enterprise / Local Jupyter / Vertex AI ---
    print(f"ℹ️ Standard Colab setup skipped ({e}). Falling back to Enterprise/Local Auth...")

    # 1. Retrieve Environment Authentication & Project ID automatically
    credentials, GCP_PROJECT_ID = google.auth.default()
    if not GCP_PROJECT_ID:
        GCP_PROJECT_ID = input("☁️ Please enter your Google Cloud Project ID: ").strip()
    print(f"✅ Authenticated via default credentials to Project: {GCP_PROJECT_ID}")

    # 2. Securely input the Google Maps API Key
    print("🔑 Please paste your Google Maps Platform API Key and press Enter:")
    GMP_API_KEY = getpass.getpass()

    # 3. Initialize BigQuery client with environment credentials
    client = bigquery.Client(credentials=credentials, project=GCP_PROJECT_ID)

print("✅ BigQuery Client Initialized.")

### Why `PLACES_COUNT_PER_H3` at High H3 Resolution Unlocks Exact Openings & Closings

> **🎯 Strategic Insight:** When analyzing a specific category (such as `coffee_shop`, `restaurant`, or `electric_vehicle_charging_station`) at **H3 Resolution 9** ($\sim 0.105\text{ km}^2$ per cell, or roughly two city blocks), the number of matching establishments in any single hexagonal cell is typically well below the **250-ID cap** returned in `sample_place_ids`.
>
> Because `sample_place_ids` contains the **complete set of active Place IDs** in every H3 Resolution 9 cell for a given `snapshot_date`, we can query `PLACES_COUNT_PER_H3` across a sequence of monthly snapshots (`'YYYY-MM'`) and compare `sample_place_ids` between Month $t-1$ and Month $t$ per `h3_cell_index`:
> * **Newly Opened Locations (`opened_place_ids`):** Place IDs present in Month $t$ that were absent in Month $t-1$.
> * **Closed Locations (`closed_place_ids`):** Place IDs present in Month $t-1$ that are no longer present in Month $t$.
> * **Zero Scratch Tables Required:** Everything executes in a single read-only query against `PLACES_COUNT_PER_H3`.

In [ ]:
# @title 2. Pipeline Configuration & Historical Window
# @markdown Keep the defaults (**London, UK** and **`coffee_shop`** at **H3 Resolution 9**) or update the parameters below to analyze any other subscribed country dataset, city coordinates, or place type.

TARGET_LOCATION_NAME = "London, UK" # @param {type:"string"}
LINKED_DATASET = "places_insights___gb" # @param ["places_insights___gb", "places_insights___gb___sample", "places_insights___us", "places_insights___us___sample", "places_insights___ca", "places_insights___au", "places_insights___de", "places_insights___fr"] {allow-input: true}
PLACE_TYPE = "coffee_shop" # @param ["coffee_shop", "cafe", "bakery", "bar", "restaurant", "grocery_store", "electric_vehicle_charging_station", "gym"] {allow-input: true}
H3_RESOLUTION = 9 # @param {type:"slider", min:7, max:10, step:1}
START_MONTH = "2025-09" # @param ["2024-01", "2024-06", "2025-01", "2025-06", "2025-09", "2026-01"] {allow-input: true}
END_MONTH = "2026-08" # @param ["2026-03", "2026-06", "2026-07", "2026-08"] {allow-input: true}
CENTER_LAT = 51.5136 # @param {type:"number"}
CENTER_LNG = -0.1196 # @param {type:"number"}
SEARCH_RADIUS_METERS = 4500 # @param {type:"slider", min:1500, max:12000, step:500}

# Generate ordered list of YYYY-MM strings from START_MONTH to END_MONTH
SNAPSHOT_MONTHS = (
    pd.date_range(start=f"{START_MONTH}-01", end=f"{END_MONTH}-01", freq="MS")
    .strftime("%Y-%m")
    .tolist()
)

print(f"📍 Target Area: {TARGET_LOCATION_NAME} ({CENTER_LAT}, {CENTER_LNG} | Radius: {SEARCH_RADIUS_METERS}m)")
print(f"🗄️ Linked Dataset: `{GCP_PROJECT_ID}.{LINKED_DATASET}`")
print(f"🏷️ Target Place Type: `{PLACE_TYPE}` at H3 Resolution {H3_RESOLUTION}")
print(f"🗓️ Monthly Snapshots ({len(SNAPSHOT_MONTHS)} months): {SNAPSHOT_MONTHS[0]} ➔ {SNAPSHOT_MONTHS[-1]}")

In [ ]:
# @title 3. Backend Initialization: Session, Copyright & Assets
# @markdown Manages the API handshake for Google Maps 2D Tiles: session creation, dynamic attribution, and branding assets.

import requests
import folium
from branca.element import Element

# Center coordinates for initial viewport from Step 2 configuration
MAP_CENTER_LAT = CENTER_LAT
MAP_CENTER_LNG = CENTER_LNG

# --- 1. Create Google Maps Tile Session ---
print("🗺️ Initializing Google Maps Session...")
session_url = f"https://tile.googleapis.com/v1/createSession?key={GMP_API_KEY}"
headers = {"Content-Type": "application/json"}
payload = {
    "mapType": "roadmap",
    "language": "en-US",
    "region": "US"
}

try:
    response = requests.post(session_url, json=payload, headers=headers, timeout=10)
    response.raise_for_status()
    session_token = response.json().get("session")
    print(f"✅ Session Token acquired: {session_token[:10]}...")
except Exception as e:
    print(f"⚠️ Warning: Could not initialize live Google Maps Tile session ({e}). Check GMP_API_KEY and Map Tiles API enablement.")
    session_token = "DEMO_SESSION_TOKEN"

# --- 2. Fetch Dynamic Attribution ---
delta = 0.05
north, south = MAP_CENTER_LAT + delta, MAP_CENTER_LAT - delta
east, west = MAP_CENTER_LNG + delta, MAP_CENTER_LNG - delta
zoom_level_attr = 13

viewport_url = (
    f"https://tile.googleapis.com/tile/v1/viewport?key={GMP_API_KEY}"
    f"&session={session_token}"
    f"&zoom={zoom_level_attr}"
    f"&north={north}&south={south}"
    f"&west={west}&east={east}"
)

try:
    vp_response = requests.get(viewport_url, timeout=10)
    vp_response.raise_for_status()
    google_attribution = vp_response.json().get('copyright', 'Map data © Google')
    print("✅ Attribution fetched.")
except Exception as e:
    print(f"⚠️ Warning: Could not fetch attribution ({e}). Defaulting.")
    google_attribution = "Map data © Google"

# --- 3. Construct Assets & Helpers ---
tiles_template_url = f"https://tile.googleapis.com/v1/2dtiles/{{z}}/{{x}}/{{y}}?session={session_token}&key={GMP_API_KEY}"
logo_url = "https://maps.gstatic.com/mapfiles/api-3/images/google_white3.png"

logo_html = f"""
    <div style="
        position: fixed;
        bottom: 35px;
        left: 10px;
        z-index: 9999;
        font-size: 0;
        pointer-events: none;
    ">
        <img src="{logo_url}" alt="Google Maps" style="height: 18px;">
    </div>
"""
print("✅ Map Assets prepared.")

### Step 4: Extracting Monthly H3 Snapshots & Computing Exact Openings/Closings in BigQuery

To compute month-over-month POI openings and closings across our target search area (`TARGET_LOCATION_NAME`) for `PLACE_TYPE`, we construct a BigQuery query that:
1. Calls **`PLACES_COUNT_PER_H3`** for each `snapshot_date` (`YYYY-MM`) in `SNAPSHOT_MONTHS` over the buffered target area (`ST_BUFFER(ST_GEOGPOINT(lon, lat), radius)`).
2. Uses the `LAG()` window function partitioned by `h3_cell_index` and ordered by `snapshot_month` to retrieve the prior month's `count` and `sample_place_ids` array (`prev_place_ids`).
3. Computes exact set differences using `ARRAY(SELECT ... FROM UNNEST(...))`:
   * **`opened_place_ids`**: IDs in `sample_place_ids` not present in `prev_place_ids` (`openings_count = ARRAY_LENGTH(opened_place_ids)`).
   * **`closed_place_ids`**: IDs in `prev_place_ids` no longer present in `sample_place_ids` (`closings_count = ARRAY_LENGTH(closed_place_ids)`).

> **💡 Production Best Practice:** Because `PLACES_COUNT_PER_H3` accepts the bounding `geography` inside a `JSON_OBJECT` subject to a **1 MB parameter size limit**, always use `ST_BUFFER(ST_GEOGPOINT(...), radius)` or wrap complex municipal boundaries in `ST_SIMPLIFY(geom, 100)` before passing them to the function.

In [ ]:
# @title 4. Query Monthly POI Snapshots & Churn with PLACES_COUNT_PER_H3
# @markdown Executes `PLACES_COUNT_PER_H3` across every month in `SNAPSHOT_MONTHS` for the configured location (`TARGET_LOCATION_NAME`) and `PLACE_TYPE`, deriving exact openings (`opened_place_ids`) and closings (`closed_place_ids`) per H3 cell.

# 1. Build a UNION ALL CTE calling PLACES_COUNT_PER_H3 for each snapshot_date
monthly_select_clauses = []
for month_str in SNAPSHOT_MONTHS:
    clause = f"""
    SELECT
      '{month_str}' AS snapshot_month,
      h3_cell_index,
      ST_ASTEXT(geography) AS geography_wkt,
      count AS active_count,
      sample_place_ids
    FROM `{GCP_PROJECT_ID}.{LINKED_DATASET}.PLACES_COUNT_PER_H3`(
      JSON_OBJECT(
        'geography', ST_BUFFER(ST_GEOGPOINT({CENTER_LNG}, {CENTER_LAT}), {SEARCH_RADIUS_METERS}),
        'h3_resolution', {H3_RESOLUTION},
        'types', ['{PLACE_TYPE}'],
        'business_status', ['OPERATIONAL'],
        'snapshot_date', '{month_str}'
      )
    )
    """
    monthly_select_clauses.append(clause.strip())

union_all_snapshots_sql = "\n    UNION ALL\n    ".join(monthly_select_clauses)

h3_historical_churn_sql = f"""
WITH monthly_h3_snapshots AS (
    {union_all_snapshots_sql}
),
windowed_h3 AS (
    SELECT
      snapshot_month,
      h3_cell_index,
      geography_wkt,
      active_count,
      sample_place_ids,
      LAG(snapshot_month) OVER (PARTITION BY h3_cell_index ORDER BY snapshot_month) AS prev_month,
      LAG(active_count) OVER (PARTITION BY h3_cell_index ORDER BY snapshot_month) AS prev_count,
      LAG(sample_place_ids) OVER (PARTITION BY h3_cell_index ORDER BY snapshot_month) AS prev_place_ids
    FROM monthly_h3_snapshots
)
SELECT
  snapshot_month,
  prev_month,
  h3_cell_index,
  geography_wkt,
  prev_count,
  active_count,
  (active_count - prev_count) AS net_change,
  -- Exact Place IDs of POIs that OPENED in this H3 cell during snapshot_month
  ARRAY(
    SELECT id FROM UNNEST(sample_place_ids) AS id
    WHERE id NOT IN UNNEST(IFNULL(prev_place_ids, ARRAY<STRING>[]))
  ) AS opened_place_ids,
  -- Exact Place IDs of POIs that CLOSED in this H3 cell during snapshot_month
  ARRAY(
    SELECT id FROM UNNEST(IFNULL(prev_place_ids, ARRAY<STRING>[])) AS id
    WHERE id NOT IN UNNEST(sample_place_ids)
  ) AS closed_place_ids
FROM windowed_h3
WHERE prev_month IS NOT NULL
ORDER BY snapshot_month ASC, h3_cell_index ASC;
"""

print(f"🚀 Executing BigQuery historical H3 query for `{PLACE_TYPE}` in {TARGET_LOCATION_NAME}...")
df_h3_trends = client.query(h3_historical_churn_sql).to_dataframe()
df_h3_trends["openings_count"] = df_h3_trends["opened_place_ids"].apply(lambda x: len(x) if x is not None else 0)
df_h3_trends["closings_count"] = df_h3_trends["closed_place_ids"].apply(lambda x: len(x) if x is not None else 0)
print(f"✅ Retrieved {len(df_h3_trends):,} H3 cell-month records across {df_h3_trends['h3_cell_index'].nunique():,} H3 cells in {TARGET_LOCATION_NAME}.")

display(df_h3_trends[["snapshot_month", "h3_cell_index", "prev_count", "active_count", "openings_count", "closings_count", "net_change"]].head(10))

### Step 5: Visualizing Month-by-Month POI Openings vs. Closings

Aggregating our H3 cell deltas by `snapshot_month` reveals the macro commercial pulse across `TARGET_LOCATION_NAME` for `PLACE_TYPE`:
* **Gross Monthly Openings (`+openings_count` in Google Green `#34A853`)**: Newly added Place IDs that appeared across H3 cells in that month.
* **Gross Monthly Closings (`-closings_count` in Google Red `#EA4335`)**: Place IDs present in the prior month that ceased operating or were removed.
* **Monthly Net Change (`net_change` in Google Blue `#1A73E8`)**: The net balance of openings minus closings each month.

In [ ]:
# @title 5. Visualize Month-by-Month POI Openings, Closings & Net Trend
# @markdown Renders an interactive diverging bar chart of monthly POI openings vs. closings overlaid with net change and total active count for `TARGET_LOCATION_NAME`.

import plotly.graph_objects as go
from plotly.subplots import make_subplots

df_monthly_summary = (
    df_h3_trends.groupby("snapshot_month", as_index=False)
    .agg(
        total_active=("active_count", "sum"),
        total_openings=("openings_count", "sum"),
        total_closings=("closings_count", "sum"),
        net_change=("net_change", "sum"),
    )
    .sort_values("snapshot_month")
)
df_monthly_summary["neg_closings"] = -df_monthly_summary["total_closings"]

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.10,
    row_heights=[0.65, 0.35],
    subplot_titles=(
        f"{TARGET_LOCATION_NAME} — `{PLACE_TYPE}` Monthly Gross Openings (+), Gross Closings (-) & Net Change (H3 Res {H3_RESOLUTION})",
        f"Total Operational `{PLACE_TYPE}` Count Across Monitored H3 Cells ({TARGET_LOCATION_NAME})",
    ),
)

# 1. Gross Openings Bar (+ Green)
fig.add_trace(
    go.Bar(
        x=df_monthly_summary["snapshot_month"],
        y=df_monthly_summary["total_openings"],
        name="New Openings (+)",
        marker_color="#34A853",
        text=df_monthly_summary["total_openings"].apply(lambda v: f"+{v}"),
        textposition="outside",
    ),
    row=1,
    col=1,
)

# 2. Gross Closings Bar (- Red)
fig.add_trace(
    go.Bar(
        x=df_monthly_summary["snapshot_month"],
        y=df_monthly_summary["neg_closings"],
        name="Closings (-)",
        marker_color="#EA4335",
        text=df_monthly_summary["total_closings"].apply(lambda v: f"-{v}"),
        textposition="outside",
    ),
    row=1,
    col=1,
)

# 3. Net Change Line (Google Blue)
fig.add_trace(
    go.Scatter(
        x=df_monthly_summary["snapshot_month"],
        y=df_monthly_summary["net_change"],
        mode="lines+markers",
        name="Net Monthly Change",
        line=dict(color="#1A73E8", width=3),
        marker=dict(size=8, color="#1A73E8"),
    ),
    row=1,
    col=1,
)

# 4. Total Active POIs Trajectory (Bottom Subplot)
fig.add_trace(
    go.Scatter(
        x=df_monthly_summary["snapshot_month"],
        y=df_monthly_summary["total_active"],
        mode="lines+markers",
        name=f"Total Active ({PLACE_TYPE})",
        fill="tozeroy",
        fillcolor="rgba(66, 133, 244, 0.12)",
        line=dict(color="#4285F4", width=2.5),
    ),
    row=2,
    col=1,
)

fig.update_layout(
    barmode="relative",
    template="plotly_white",
    height=680,
    font=dict(family="Google Sans, Roboto, Arial, sans-serif", size=12, color="#202124"),
    legend=dict(orientation="h", yanchor="bottom", y=1.04, xanchor="right", x=1.0),
    margin=dict(t=80, b=50, l=60, r=40),
)
fig.update_yaxes(title_text="Monthly Openings / Closings", zeroline=True, zerolinewidth=1.5, zerolinecolor="#5F6368", row=1, col=1)
fig.update_yaxes(title_text="Total Active POIs", row=2, col=1)

fig.show()
display(df_monthly_summary[["snapshot_month", "total_active", "total_openings", "total_closings", "net_change"]])

### Step 6: Interactive H3 Spatial Map on Google Maps 2D Tiles (with Map Key)

Next, we project our H3 hexagons onto an interactive **Folium** map backed by **Google Maps 2D Roadmap Tiles** and an embedded **Map Key**. Users can select any `SELECTED_MONTH` or `All Months Cumulative` and toggle between `openings_count`, `closings_count`, `net_change`, and `active_count` to inspect where commercial expansion and churn are concentrated across `TARGET_LOCATION_NAME`.

> **💡 Production Best Practice (Folium Serialization Protection):** BigQuery array columns (`sample_place_ids`, `opened_place_ids`, `closed_place_ids`) deserialize into Python `list` or `numpy.ndarray` objects, which are unhashable and cause `TypeError` failures during Folium GeoJSON serialization. Always drop or convert array columns to formatted strings before passing a `GeoDataFrame` to `folium.Choropleth` or `folium.GeoJson`.

In [ ]:
# @title 6. Render Interactive H3 Map on Google Maps 2D Tiles
# @markdown Select a snapshot month (or `All Months Cumulative`) and a metric to render an interactive H3 choropleth map with an embedded Map Key over Google Maps 2D Tiles.

SELECTED_MONTH = "All Months Cumulative" # @param ["All Months Cumulative", "2025-10", "2025-12", "2026-03", "2026-06", "2026-08"] {allow-input: true}
VISUALIZATION_METRIC = "net_change" # @param ["openings_count", "closings_count", "net_change", "active_count"]

# 1. Filter or aggregate the H3 DataFrame based on SELECTED_MONTH
if SELECTED_MONTH == "All Months Cumulative" or SELECTED_MONTH not in df_h3_trends["snapshot_month"].values:
    df_map_slice = (
        df_h3_trends.sort_values("snapshot_month")
        .groupby("h3_cell_index", as_index=False)
        .agg(
            geography_wkt=("geography_wkt", "last"),
            prev_count=("prev_count", "first"),
            active_count=("active_count", "last"),
            openings_count=("openings_count", "sum"),
            closings_count=("closings_count", "sum"),
            net_change=("net_change", "sum"),
        )
    )
    period_label = f"{SNAPSHOT_MONTHS[0]} to {SNAPSHOT_MONTHS[-1]} (Cumulative)"
else:
    # Drop unhashable list/array columns for Folium GeoJSON serialization safety
    df_map_slice = (
        df_h3_trends[df_h3_trends["snapshot_month"] == SELECTED_MONTH]
        .drop(columns=["opened_place_ids", "closed_place_ids"], errors="ignore")
        .copy()
    )
    period_label = f"Month: {SELECTED_MONTH}"

# 2. Convert WKT strings to Shapely geometries and build GeoDataFrame
df_map_slice["geometry"] = df_map_slice["geography_wkt"].apply(wkt.loads)
gdf_h3 = gpd.GeoDataFrame(df_map_slice, geometry="geometry", crs="EPSG:4326")

# 3. Initialize Folium Map with Google Maps 2D Tiles & Dynamic Attribution
m = folium.Map(
    location=[MAP_CENTER_LAT, MAP_CENTER_LNG],
    zoom_start=13,
    tiles=tiles_template_url,
    attr=google_attribution,
)

# 4. Choose color scale and Map Key palette matching metric semantics
metric_metadata = {
    "openings_count": {
        "cmap": "Greens",
        "title": "New Openings (+)",
        "swatches": ["#E6F4EA", "#A8DAB5", "#5BB974", "#1E8E3E", "#0D652D"],
        "low_label": "0 (No Openings)",
        "high_label": "High Openings Hotspot",
    },
    "closings_count": {
        "cmap": "Reds",
        "title": "Closings / Removals (-)",
        "swatches": ["#FCE8E6", "#F6AEA9", "#EE675C", "#D93025", "#A50E0E"],
        "low_label": "0 (No Closings)",
        "high_label": "High Churn / Closings",
    },
    "net_change": {
        "cmap": "RdYlGn",
        "title": "Net Change (Openings - Closings)",
        "swatches": ["#D93025", "#F6AEA9", "#FEF7E0", "#81C995", "#1E8E3E"],
        "low_label": "Net Contraction (-)",
        "high_label": "Net Expansion (+)",
    },
    "active_count": {
        "cmap": "YlGnBu",
        "title": "Total Active POIs",
        "swatches": ["#FFFFD9", "#C7E9B4", "#41B6C4", "#225EA8", "#081D58"],
        "low_label": "Low Density",
        "high_label": "High Concentration",
    },
}
meta = metric_metadata.get(VISUALIZATION_METRIC, metric_metadata["openings_count"])
min_val = int(gdf_h3[VISUALIZATION_METRIC].min()) if len(gdf_h3) else 0
max_val = int(gdf_h3[VISUALIZATION_METRIC].max()) if len(gdf_h3) else 0
mid_val = round((min_val + max_val) / 2, 1)

folium.Choropleth(
    geo_data=gdf_h3.to_json(),
    name=f"H3 Res {H3_RESOLUTION} - {meta['title']}",
    data=gdf_h3,
    columns=["h3_cell_index", VISUALIZATION_METRIC],
    key_on="feature.properties.h3_cell_index",
    fill_color=meta["cmap"],
    fill_opacity=0.68,
    line_opacity=0.85,
    line_color="#202124",
    legend_name=f"{TARGET_LOCATION_NAME} `{PLACE_TYPE}` — {meta['title']} ({period_label})",
).add_to(m)

# 5. Add interactive hover tooltips per H3 cell
tooltip_fields = ["h3_cell_index", "active_count", "openings_count", "closings_count", "net_change"]
tooltip_aliases = [
    f"H3 Index (Res {H3_RESOLUTION}):",
    f"Active ({PLACE_TYPE}):",
    "New Openings (+):",
    "Closings (-):",
    "Net Change:",
]

folium.GeoJson(
    gdf_h3,
    name="H3 Cell Inspector",
    style_function=lambda _: {"fillOpacity": 0.0, "color": "transparent", "weight": 0},
    highlight_function=lambda _: {"fillOpacity": 0.25, "color": "#1A73E8", "weight": 2.5},
    tooltip=folium.GeoJsonTooltip(
        fields=tooltip_fields,
        aliases=tooltip_aliases,
        localize=True,
        sticky=False,
        style="font-family: 'Google Sans', Roboto, sans-serif; font-size: 12px; padding: 8px;",
    ),
).add_to(m)

# 6. Inject Floating Map Key (Legend) Overlay + Official Google Maps Logo Overlay
swatches_html = "".join(
    [f'<span style="flex:1; height:10px; background:{c}; display:inline-block;"></span>' for c in meta["swatches"]]
)
map_key_html = f"""
<div style="
    position: fixed;
    bottom: 35px;
    right: 14px;
    z-index: 9999;
    background: rgba(255, 255, 255, 0.96);
    border: 1px solid #DADCE0;
    border-radius: 10px;
    padding: 12px 14px;
    width: 265px;
    box-shadow: 0 2px 8px rgba(32, 33, 36, 0.14);
    font-family: 'Google Sans', Roboto, Arial, sans-serif;
    color: #202124;
    font-size: 12px;
    line-height: 1.4;
">
    <div style="font-weight: 700; font-size: 13px; margin-bottom: 2px;">
        🗺️ Map Key — {meta['title']}
    </div>
    <div style="color: #5F6368; font-size: 11px; margin-bottom: 8px;">
        {TARGET_LOCATION_NAME} • <code>{PLACE_TYPE}</code> • {period_label}
    </div>
    <div style="display: flex; border-radius: 4px; overflow: hidden; border: 1px solid #DADCE0; margin-bottom: 4px;">
        {swatches_html}
    </div>
    <div style="display: flex; justify-content: space-between; font-size: 11px; color: #3C4043; font-weight: 600; margin-bottom: 4px;">
        <span>Min: {min_val}</span>
        <span>Mid: {mid_val}</span>
        <span>Max: {max_val}</span>
    </div>
    <div style="display: flex; justify-content: space-between; font-size: 10px; color: #5F6368; margin-bottom: 8px;">
        <span>{meta['low_label']}</span>
        <span>{meta['high_label']}</span>
    </div>
    <hr style="border: none; border-top: 1px solid #E8EAED; margin: 6px 0;">
    <div style="font-size: 11px; color: #3C4043;">
        <span style="display:inline-block; width:10px; height:10px; border:1.5px solid #202124; background:rgba(66,133,244,0.15); vertical-align:middle; margin-right:5px;"></span>
        <b>H3 Res {H3_RESOLUTION} Hexagon</b> (~0.105 km²)<br>
        <span style="display:inline-block; width:10px; height:10px; border:2px solid #1A73E8; background:rgba(26,115,232,0.25); vertical-align:middle; margin-right:5px; margin-top:3px;"></span>
        <b>Hover Cell</b> for exact counts & deltas
    </div>
</div>
"""

m.get_root().html.add_child(Element(logo_html))
m.get_root().html.add_child(Element(map_key_html))
folium.LayerControl().add_to(m)

print(f"🗺️ Rendering H3 Resolution {H3_RESOLUTION} Map for {TARGET_LOCATION_NAME} ({period_label}) | Metric: `{VISUALIZATION_METRIC}`")
display(m)

### Step 7: Ground-Truth Verification — Inspecting Newly Opened & Closed POIs via Places API (New)

While `PLACES_COUNT_PER_H3` intentionally omits storefront names and phone numbers inside BigQuery for privacy and governance, the `opened_place_ids` and `closed_place_ids` arrays we derived from `sample_place_ids` let us transition directly from **macro statistical discovery** to **location-specific ground truth**.

In the cell below, we extract the top H3 cells in `TARGET_LOCATION_NAME` with the highest `PLACE_TYPE` turnover and pass their `opened_place_ids` and `closed_place_ids` to the **[Places API (New)](https://developers.google.com/maps/documentation/places/web-service/place-details)** (`GET https://places.googleapis.com/v1/places/{place_id}`) to inspect the exact businesses that opened or closed.

> **💡 Production Best Practice:** When enriching hundreds of Place IDs in production, always specify a tight `X-Goog-FieldMask` header (e.g., `id,displayName,formattedAddress,rating,businessStatus`) to minimize response payload size and control Places API SKU billing tiers.

In [ ]:
# @title 7. Enrich Opened & Closed Place IDs via Google Maps Places API (New)
# @markdown Fetches ground-truth storefront names, addresses, and ratings for newly opened and closed `PLACE_TYPE` locations in high-turnover H3 cells across `TARGET_LOCATION_NAME`.

MAX_PLACES_TO_INSPECT = 6 # @param {type:"slider", min:2, max:15, step:1}

# Collect sample opened and closed Place IDs from the DataFrame
sample_events = []
for _, row in df_h3_trends.sort_values(["openings_count", "closings_count"], ascending=False).iterrows():
    for pid in (row["opened_place_ids"] if row["opened_place_ids"] is not None else []):
        if len(sample_events) < MAX_PLACES_TO_INSPECT:
            sample_events.append(("🟢 NEW OPENING", row["snapshot_month"], row["h3_cell_index"], pid))
    for pid in (row["closed_place_ids"] if row["closed_place_ids"] is not None else []):
        if len(sample_events) < MAX_PLACES_TO_INSPECT:
            sample_events.append(("🔴 CLOSED / REMOVED", row["snapshot_month"], row["h3_cell_index"], pid))
    if len(sample_events) >= MAX_PLACES_TO_INSPECT:
        break

def fetch_place_details(place_id: str, api_key: str) -> dict:
    """Calls Places API (New) to retrieve ground-truth storefront metadata."""
    url = f"https://places.googleapis.com/v1/places/{place_id}"
    headers = {
        "X-Goog-Api-Key": api_key,
        "X-Goog-FieldMask": "id,displayName,formattedAddress,rating,businessStatus",
    }
    resp = requests.get(url, headers=headers, timeout=10)
    resp.raise_for_status()
    data = resp.json()
    return {
        "displayName": data.get("displayName", {}).get("text", "Unknown"),
        "formattedAddress": data.get("formattedAddress", "N/A"),
        "rating": data.get("rating", None),
        "businessStatus": data.get("businessStatus", "UNKNOWN"),
    }

enriched_records = []
for event_type, month_val, h3_cell, pid in sample_events[:MAX_PLACES_TO_INSPECT]:
    try:
        details = fetch_place_details(pid, GMP_API_KEY)
        enriched_records.append({
            "Event Type": event_type,
            "Snapshot Month": month_val,
            f"H3 Cell (Res {H3_RESOLUTION})": h3_cell,
            "Place ID": pid,
            "Storefront Name": details["displayName"],
            "Address": details["formattedAddress"],
            "Rating": details["rating"],
            "Current Status": details["businessStatus"],
        })
    except Exception as api_err:
        enriched_records.append({
            "Event Type": event_type,
            "Snapshot Month": month_val,
            f"H3 Cell (Res {H3_RESOLUTION})": h3_cell,
            "Place ID": pid,
            "Storefront Name": f"API Lookup Note ({str(api_err)[:40]})",
            "Address": "Verify Places API (New) enablement",
            "Rating": None,
            "Current Status": "N/A",
        })

df_ground_truth = pd.DataFrame(enriched_records)
print(f"✅ Enriched {len(df_ground_truth)} ground-truth `{PLACE_TYPE}` opening/closing events in {TARGET_LOCATION_NAME}:")
display(df_ground_truth)